# 투수-타자 맞대결 피처 실험 노트북

`train_ensemble.py`에 새로 추가한 `build_matchup_features`(as-of 맞대결 성공률 +
표본 수 기반 shrinkage)가 실제로 LightGBM 성능을 개선하는지 확인합니다.

**사전 검증 결과** (구현 전에 확인함):
- `row_id` 순서 = 실제 시간순 (pitcher_id별 `asof_pitcher_n`이 단조증가, 역행 0건)
- 전체 행의 **93.5%**가 이미 해당 (투수,타자) 조합의 재대결 (재대결 행 기준 중앙값 11회)
- `matchup_success_rate_shrunk`와 `control_success`의 상관계수: **0.0873**
  (참고 - 기존 `asof_pitcher_success_rate`: 0.0843) → 기존 피처보다 살짝 더 강한 신호

**비교 기준선** (기존 `randomforest.ipynb`에서 확인한 값):
- LightGBM 20만행 OOF Brier: **0.245666**
- LightGBM 전체(147만행) OOF Brier: **0.244139**

이 노트북은 새 피처를 포함해서 다시 학습했을 때 이 값보다 낮아지는지 확인합니다.

**이 파일과 `train_ensemble.py`는 같은 폴더에 있어야 아래 import가 동작합니다.**

In [8]:
import sys, os, time
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

sys.path.append(os.getcwd())
from train_ensemble import (
    TARGET_COL, CAT_COLS, build_features, train_lgb,
)

DATA_DIR = "../open/data"

## 1. 데이터 로드 & 피처 생성 (맞대결 피처 포함)

`build_features`가 내부적으로 `build_matchup_features`를 자동 호출하므로
별도 작업 없이 바로 `matchup_*` 컬럼이 포함된 `feat_cols`가 만들어집니다.

In [10]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
print(train.shape)

train_feat, feat_cols = build_features(train, None)
cat_features = [c for c in CAT_COLS if c in feat_cols]

matchup_cols = [c for c in feat_cols if c.startswith("matchup")]
print(f"피처 개수: {len(feat_cols)} (맞대결 피처 {len(matchup_cols)}개 포함: {matchup_cols})")

r = train_feat[TARGET_COL].mean()
baseline_brier = r * (1 - r)
print(f"기준(무정보) Brier = {baseline_brier:.5f}")

(1475092, 49)
피처 개수: 74 (맞대결 피처 5개 포함: ['matchup_n', 'matchup_n_log', 'matchup_success_rate', 'matchup_success_rate_isna', 'matchup_success_rate_shrunk'])
기준(무정보) Brier = 0.24944


## 2. 20만행 샘플로 빠른 확인

기존 실험과 동일한 조건(`random_state=42`, 20만행)으로 샘플링해서
LightGBM 기준선(0.245666)과 직접 비교합니다.

In [11]:
SAMPLE_N = 200_000

sample_idx = train_feat.sample(n=min(SAMPLE_N, len(train_feat)), random_state=42).index
X_small = train_feat.loc[sample_idx, feat_cols]
y_small = train_feat.loc[sample_idx, TARGET_COL].values

t0 = time.time()
lgb_models_mu, lgb_oof_mu = train_lgb(X_small, y_small, X_small, cat_features)
brier_mu = brier_score_loss(y_small, lgb_oof_mu)
print(f"[LightGBM+맞대결] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+맞대결] OOF Brier (20만행): {brier_mu:.5f}")
print(f"참고 - 맞대결 피처 없는 기존 LightGBM 20만행 Brier: 0.245666")
print(f"개선폭: {(0.245666 - brier_mu) / 0.245666 * 100:.4f}% (양수면 개선)")

  [LGB fold 0] brier=0.24564
  [LGB fold 1] brier=0.24572
  [LGB fold 2] brier=0.24488
  [LGB fold 3] brier=0.24574
  [LGB fold 4] brier=0.24582
[LightGBM+맞대결] 소요시간: 18.7초
[LightGBM+맞대결] OOF Brier (20만행): 0.24556
참고 - 맞대결 피처 없는 기존 LightGBM 20만행 Brier: 0.245666
개선폭: 0.0426% (양수면 개선)


## 3. 맞대결 피처의 LightGBM Feature Importance 확인

20만행 결과가 개선됐다면, 실제로 트리가 이 피처를 얼마나 활용하는지 확인합니다.
`share_pct`가 낮으면(예: 1% 미만) 있어도 그만 없어도 그만인 피처일 수 있습니다.

In [12]:
imp_df = pd.DataFrame({
    f"fold{i}": m.feature_importance(importance_type="gain")
    for i, m in enumerate(lgb_models_mu)
}, index=feat_cols)
imp_df["mean_gain"] = imp_df[[c for c in imp_df.columns if c.startswith("fold")]].mean(axis=1)
imp_df["share_pct"] = imp_df["mean_gain"] / imp_df["mean_gain"].sum() * 100
imp_df = imp_df.sort_values("mean_gain", ascending=False)

print("전체 피처 중 순위:")
imp_df_ranked = imp_df.reset_index().rename(columns={"index": "feature"})
imp_df_ranked["rank"] = imp_df_ranked.index + 1
display(imp_df_ranked[imp_df_ranked["feature"].isin(matchup_cols)][["rank", "feature", "mean_gain", "share_pct"]])

print(f"\n참고 - 상위 10개 피처:")
display(imp_df[["mean_gain", "share_pct"]].head(10))

전체 피처 중 순위:


,rank,feature,mean_gain,share_pct
0,1,matchup_success_rate_shrunk,10315.099838,9.710693
11,12,matchup_success_rate,3063.562688,2.884055
27,28,matchup_n,1296.944986,1.220951
45,46,matchup_n_log,295.349386,0.278044
58,59,matchup_success_rate_isna,0.000000,0.000000



참고 - 상위 10개 피처:


,mean_gain,share_pct
matchup_success_rate_shrunk,10315.099838,9.710693
asof_pitcher_success_rate,8810.117967,8.293895
season,6306.692286,5.937156
game_type,4438.023288,4.177980
asof_batter_success_rate,4275.039340,4.024546
asof_pitcher_prev3_game_success_rate,4199.144565,3.953098
asof_pitcher_reverse_rate,4094.550124,3.854633
asof_pitcher_prev5_game_success_rate,3499.146604,3.294116
asof_pitcher_prev1_game_success_rate,3285.748014,3.093222
asof_pitcher_ball_rate,3193.973880,3.006825


## 4. (20만행 결과가 개선됐을 때만 실행) 전체 데이터로 확장

20만행에서 개선이 확인되면 전체 147만행으로 다시 학습해서
기존 체크포인트의 LightGBM 전체 결과(0.244139)와 비교합니다.

In [13]:
X_full = train_feat[feat_cols]
y_full = train_feat[TARGET_COL].values

t0 = time.time()
lgb_models_mu_full, lgb_oof_mu_full = train_lgb(X_full, y_full, X_full, cat_features)
brier_mu_full = brier_score_loss(y_full, lgb_oof_mu_full)
print(f"[LightGBM+맞대결] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+맞대결] OOF Brier (전체): {brier_mu_full:.5f}")
print(f"참고 - 맞대결 피처 없는 기존 LightGBM 전체 Brier: 0.244139")
print(f"개선폭: {(0.244139 - brier_mu_full) / 0.244139 * 100:.4f}% (양수면 개선)")

  [LGB fold 0] brier=0.24427
  [LGB fold 1] brier=0.24430
  [LGB fold 2] brier=0.24431
  [LGB fold 3] brier=0.24433
  [LGB fold 4] brier=0.24431
[LightGBM+맞대결] 소요시간: 250.1초
[LightGBM+맞대결] OOF Brier (전체): 0.24430
참고 - 맞대결 피처 없는 기존 LightGBM 전체 Brier: 0.244139
개선폭: -0.0678% (양수면 개선)


## 5. shrink_k를 키워서 재검증 (k=10 -> k=100)

20만행에서는 개선(+0.043%)처럼 보였지만 전체 데이터에서는 악화(-0.068%)로 뒤집혔습니다.
20만행 결과가 신뢰할 수 없다는 게 이미 확인됐으니, 이번에는 20만행 단계를 건너뛰고
**바로 전체 데이터로** k=100(더 강한 축소추정, 저표본 맞대결의 노이즈를 더 죽임)을 확인합니다.

`build_features`에 `matchup_shrink_k` 파라미터를 추가해뒀으므로 원본 `train`에서
다시 피처를 만들기만 하면 됩니다(`train_feat`을 덮어쓰지 않고 별도 변수로 유지).

In [18]:
import importlib, train_ensemble
importlib.reload(train_ensemble)
from train_ensemble import build_features

In [19]:
train_feat_k100, feat_cols_k100 = build_features(train, None, matchup_shrink_k=100)
cat_features_k100 = [c for c in CAT_COLS if c in feat_cols_k100]

X_full_k100 = train_feat_k100[feat_cols_k100]
y_full_k100 = train_feat_k100[TARGET_COL].values

t0 = time.time()
lgb_models_k100, lgb_oof_k100 = train_lgb(X_full_k100, y_full_k100, X_full_k100, cat_features_k100)
brier_k100 = brier_score_loss(y_full_k100, lgb_oof_k100)
print(f"[LightGBM+맞대결 k=100] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+맞대결 k=100] OOF Brier (전체): {brier_k100:.5f}")
print(f"참고 - 맞대결 피처 없음(전체): 0.244139")
print(f"참고 - 맞대결 k=10(전체): 0.24430")
print(f"k=100 개선폭 (vs 피처 없음): {(0.244139 - brier_k100) / 0.244139 * 100:.4f}% (양수면 개선)")

  [LGB fold 0] brier=0.24429
  [LGB fold 1] brier=0.24431
  [LGB fold 2] brier=0.24430
  [LGB fold 3] brier=0.24433
  [LGB fold 4] brier=0.24430
[LightGBM+맞대결 k=100] 소요시간: 236.5초
[LightGBM+맞대결 k=100] OOF Brier (전체): 0.24430
참고 - 맞대결 피처 없음(전체): 0.244139
참고 - 맞대결 k=10(전체): 0.24430
k=100 개선폭 (vs 피처 없음): -0.0676% (양수면 개선)
